#### Capstone Project

**THE PRICE IS RIGHT**

1. Data Curation

-> 2. **Data Pre-Processing**

3. Evaluation against Baselines and Traditional ML

4. Deep Learning and LLMs

5. Fine-Tuning a Frontier Model

In [ ]:
# to move one folder up for pricer folder

import os
import sys

sys.path.insert(0,os.path.abspath('..'))

In [3]:
# imports

from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

In [ ]:
# selecting the dataset

LITE_MODE = True

In [ ]:
# loading the dataset

user_name = 'lalam0'

dataset = f"{user_name}/items_raw_lite" if LITE_MODE else f"{user_name}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f'Loaded {len(items):,} items')

README.md:   0%|          | 0.00/738 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.8MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.04MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 22,000 items


In [6]:
# sample datapoint

items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [ ]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [8]:
# give every item an ID

for index, item in enumerate(items):
    item.id = index

In [ ]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [ ]:
# system prompt

SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [11]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [ ]:
# calling gpt-oss-20b to pre-process the first element in the dataset

messages = [{'role':'system', 'content':SYSTEM_PROMPT}, {'role':'user', 'content':items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f'Input Tokens: {response.usage.prompt_tokens}')
print(f'Ouptut Tokens: {response.usage.completion_tokens}')
print(f'Cost: {response._hidden_params['response_cost']*100:.3f} cents')

Title: Schlage Interior Knob with Deadbolt, Oil‑Rubbed Bronze  
Category: Security Hardware  
Brand: Schlage  
Description: An oil‑rubbed bronze interior knob paired with a deadbolt for secure, easy‑to‑install door access.  
Details: Features a solid metal construction, 1.5‑lb weight, 8.1×4.4×3.7 in dimensions, and a lifetime mechanical and finish warranty.

Input Tokens: 446
Ouptut Tokens: 112
Cost: 0.007 cents


In [14]:
# calling llama3.2 via ollama

messages = [{'role':'system', 'content':SYSTEM_PROMPT}, {'role':'user', 'content':items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", base_url="http://localhost:11434")

print(response.choices[0].message.content)
print()
print(f'Input Tokens: {response.usage.prompt_tokens}')
print(f'Ouptut Tokens: {response.usage.completion_tokens}')
print(f'Cost: {response._hidden_params['response_cost']*100:.3f} cents')

### Product
Title: Schlage Door Knob
Category: Home Security
Brand: Schlage
Description: Secure your home with this oil rubbed bronze door knob, engineered for precision and durability.
Details: Features a lifetime mechanical and finish warranty, ensuring years of reliable service.

Input Tokens: 406
Ouptut Tokens: 57
Cost: 0.000 cents
